# Subject-Level Evaluation for Alzheimer's MRI Classification
The same model is trained with a slice-level (leaky) and a subject-level split, cross-validated, probed with permuted labels and visualised with Grad-CAM.

**Before running:** Runtime → Change runtime type → **T4 GPU**

In [ ]:
!nvidia-smi

In [ ]:
GITHUB_USER = "mahdildarr"
!git clone https://github.com/{GITHUB_USER}/alzheimer-mri.git
%cd alzheimer-mri
!pip install -q -e . kagglehub

## 1. Dataset
Downloaded from Kaggle. If you get an access error, log in with your Kaggle account (`kagglehub.login()`).

In [ ]:
import kagglehub
DATA = kagglehub.dataset_download("ninadaithal/imagesoasis")
print(DATA)
!ls "{DATA}"

## 2. Single hold-out split: slice-level (leaky) vs subject-level

In [ ]:
!python -m admri.train --root "{DATA}" --split slice --epochs 8
!python -m admri.train --root "{DATA}" --split subject --epochs 8

## 3. Leakage probe: labels permuted between subjects
With a subject-level split this must be at chance; anything above chance with the slice-level split is subject memorisation.

In [ ]:
!python -m admri.train --root "{DATA}" --split slice --permute-labels --epochs 8
!python -m admri.train --root "{DATA}" --split subject --permute-labels --epochs 8
!python -m admri.report --runs runs/slice runs/subject runs/slice_permuted runs/subject_permuted

## 4. 5-fold cross-validation of both protocols
About 4× the time of steps 2–3; reduce `--folds` or `--epochs` if needed.

In [ ]:
!python -m admri.cv --root "{DATA}" --folds 5 --epochs 8 --permute --out runs/cv

## 5. Grad-CAM

In [ ]:
!python -m admri.gradcam --run runs/subject --n 8 --all
from IPython.display import Image, Markdown, display
display(Markdown(open("runs/report.md").read()))
display(Markdown(open("runs/cv/summary.md").read()))
display(Image("runs/subject/gradcam.png", width=1200))

## Save results
Copy the outputs to Google Drive and update the results tables in the README.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/alzheimer-mri
!cp -r runs /content/drive/MyDrive/alzheimer-mri/